## 1. Setup

In [ ]:
import os, glob, shutil, zipfile

ON_KAGGLE = os.path.exists("/kaggle/input")
WORK_DIR = "/kaggle/working/sentiment_analysis" if ON_KAGGLE else "/content/sentiment_analysis"

if not os.path.exists(WORK_DIR):
    if ON_KAGGLE:
        found = glob.glob("/kaggle/input/**/model_training.py", recursive=True)
        assert found, "model_training.py not found in /kaggle/input: attach the whole repo, not only dataset/"
        shutil.copytree(os.path.dirname(found[0]), WORK_DIR)
    else:
        from google.colab import files
        zip_name = next(iter(files.upload()))
        zipfile.ZipFile(zip_name).extractall("/content")

os.chdir(WORK_DIR)
print(os.getcwd(), sorted(os.listdir("dataset")))

In [ ]:
!pip install -q datasets

import time, json
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer,
)

from model_training import load_test_data, summarize, LABELS

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("transformers", transformers.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## 2. Data

In [ ]:
train = pd.read_csv("dataset/sentiment_train.csv")
val = pd.read_csv("dataset/sentiment_validation.csv")
test = load_test_data()

for name, df in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s} {len(df):6,} rows  {df['label'].value_counts().sort_index().tolist()}  (neg, neu, pos)")
assert not test["text"].isin(train["text"]).any(), "test leaks into train"


In [ ]:
MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True)

ds_train, ds_val, ds_test = to_dataset(train), to_dataset(val), to_dataset(test)

## 3. Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id={label: i for i, label in enumerate(LABELS)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return summarize([LABELS[i] for i in labels], [LABELS[i] for i in preds])

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
train_minutes = (time.time() - start) / 60
print(f"Training time: {train_minutes:.1f} min")

In [ ]:
print(pd.DataFrame([h for h in trainer.state.log_history if "eval_f1_macro" in h])
      [["epoch", "eval_loss", "eval_accuracy", "eval_recall_macro", "eval_f1_macro"]].round(4).to_string(index=False))

## 4. Evaluate (test set)

In [ ]:
logits = trainer.predict(ds_test).predictions
test["pred"] = [LABELS[i] for i in np.argmax(logits, axis=-1)]
metrics = summarize(test["sentiment"], test["pred"])

for k, v in metrics.items():
    print(f"{k:16s}: {v:.4f}")
print()
print(pd.DataFrame(classification_report(test["sentiment"], test["pred"], labels=LABELS, output_dict=True, zero_division=0))
      .transpose().drop(index="accuracy", errors="ignore").round(3))
print()
print(pd.DataFrame(confusion_matrix(test["sentiment"], test["pred"], labels=LABELS),
                   index=[f"actual_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS]))

## 5. Latency

In [ ]:
cpu_model = trainer.model.to("cpu").eval()
torch.set_num_threads(2)

with torch.no_grad():
    cpu_model(**tokenizer("warm up", return_tensors="pt"))
    start = time.perf_counter()
    for text in test["text"]:
        cpu_model(**tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"))
metrics["latency_ms"] = (time.perf_counter() - start) / len(test) * 1000
print(f"CPU latency: {metrics['latency_ms']:.1f} ms/sentence")

## 6. Compare with LinearSVC

In [ ]:
from model_training import train_model

_, _, _, svc_metrics = train_model()
test["pred_svc"] = svc_metrics["predictions"]

both_right = ((test.pred == test.sentiment) & (test.pred_svc == test.sentiment)).sum()
only_transformer = ((test.pred == test.sentiment) & (test.pred_svc != test.sentiment)).sum()
only_svc = ((test.pred != test.sentiment) & (test.pred_svc == test.sentiment)).sum()
print(f"LinearSVC F1 macro  : {svc_metrics['f1_macro']:.4f}")
print(f"Transformer F1 macro: {metrics['f1_macro']:.4f}\n")
print(f"Both right           : {both_right}")
print(f"Only Transformer right: {only_transformer}")
print(f"Only LinearSVC right : {only_svc}\n")

fixed = test[(test.pred == test.sentiment) & (test.pred_svc != test.sentiment)]
for _, r in fixed.sample(min(15, len(fixed)), random_state=0).iterrows():
    print(f"[{r.sentiment}: SVC {r.pred_svc}] {r.text[:150]}")

## 7. Save

In [ ]:
MODEL_DIR = "models/distilroberta-sentiment"
trainer.save_model(MODEL_DIR)

os.makedirs("results", exist_ok=True)
test[["id", "text", "sentiment", "pred"]].to_csv("results/distilroberta_test_predictions.csv", index=False)
metrics.update(model=MODEL_NAME, test_size=len(test), train_minutes=train_minutes, gpu=torch.cuda.get_device_name(0))
json.dump(metrics, open("results/distilroberta_metrics.json", "w"), indent=2)

shutil.rmtree("checkpoints", ignore_errors=True)
!zip -qr distilroberta_results.zip {MODEL_DIR} results

if ON_KAGGLE:
    print(f"Download from the Output panel: {WORK_DIR}/distilroberta_results.zip")
else:
    from google.colab import files
    files.download("distilroberta_results.zip")